# عمّان عبر الزمن — Final GPU render
**Before running:** Runtime → Change runtime type → **T4 GPU** (or better). Then run cells 1→6 in order. Progress is saved to Google Drive (`MyDrive/amman_render/frames`), so if the session disconnects just re-run cells 1, 2, 4 and it resumes where it stopped.

In [ ]:
# 1) GPU + storage check (stops here if no GPU)
import subprocess, shutil, os
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv", shell=True, capture_output=True, text=True).stdout or "NO GPU")
assert shutil.which("nvidia-smi"), "No GPU: Runtime > Change runtime type > T4 GPU, then re-run"
print("free disk GB:", shutil.disk_usage("/content").free // 2**30)
from google.colab import drive; drive.mount("/content/drive")
FR = "/content/drive/MyDrive/amman_render/frames"; os.makedirs(FR, exist_ok=True)

In [ ]:
%%bash
set -e
cd /content
[ -d roman-amman ] && (cd roman-amman && git pull -q) || git clone -q https://github.com/michobeshay/roman-amman.git
if [ ! -x /content/bl/blender ]; then
  curl -fL --retry 3 -o bl.tar.xz https://download.blender.org/release/Blender5.0/blender-5.0.1-linux-x64.tar.xz
  mkdir -p bl && tar -xf bl.tar.xz -C bl --strip-components=1 && rm bl.tar.xz
fi
apt-get -qq install -y fonts-noto-core libraqm0 libfribidi0 libxi6 libxxf86vm1 libxfixes3 libxrender1 libgl1 libxkbcommon0 libsm6 > /dev/null 2>&1 || true
pip -q install edge-tts arabic-reshaper python-bidi scipy
/content/bl/blender --version | head -1

In [ ]:
# 3) Benchmark 3 frames on this GPU -> real ETA. Tune PCT/SPP here (defaults: 67% = 724x1286, upscaled to 1080x1920 in post).
import subprocess, re, os
PCT, SPP = 67, 40
os.environ["BEE_GPU"] = "1"
r = subprocess.run(f"cd /content/roman-amman && /content/bl/blender -b -P blender/render_amman.py -- --frames 1,1500,2300 --spp {SPP} --pct {PCT} --out /content/bench", shell=True, capture_output=True, text=True)
out = r.stdout; print("\n".join(l for l in out.splitlines() if l.startswith("RS")))
s = [float(x) for x in re.findall(r"RS frame \d+ seconds ([\d.]+)", out)]
if s: avg = sum(s)/len(s); print(f"avg {avg:.1f}s/frame -> ETA {avg*3150/3600:.1f} h for 3150 frames (excludes ~10s scene build per chunk)")
else: print(r.stderr[-1500:])

In [ ]:
# 4) RENDER (resume-safe, saved to Drive). Re-run this cell after any disconnect.
import subprocess, os, glob
os.environ["BEE_GPU"] = "1"; CH = 150
def done(): return len(glob.glob(FR + "/F_*.png"))
for a in range(1, 3151, CH):
    b = min(a + CH - 1, 3150)
    if all(os.path.exists(f"{FR}/F_{i:04d}.png") for i in (a, b)) and len([1 for i in range(a, b+1) if os.path.exists(f"{FR}/F_{i:04d}.png")]) == b-a+1: continue
    p = subprocess.run(f"cd /content/roman-amman && /content/bl/blender -b -P blender/render_amman.py -- --range {a}-{b} --spp {SPP} --pct {PCT} --out {FR}", shell=True, capture_output=True, text=True)
    print(f"{a}-{b} done, total frames on Drive: {done()}", [l for l in p.stdout.splitlines() if "RS range" in l])
print("ALL DONE" if done() >= 3150 else "partial - run again")

In [ ]:
# 5) Label positions (needs a Blender pass, no rendering) + narration + audio
!cd /content/roman-amman && /content/bl/blender -b -P blender/render_amman.py -- --labels /content/labels.json --out /content/none | grep RS
!cd /content/roman-amman && python3 post/audio_amman.py /content/audio.wav /content/vo

In [ ]:
# 6) Post (grade, Arabic titles/labels/timeline/subtitles) + download
!cd /content/roman-amman && python3 post/post_amman.py {FR} /content/amman_through_time.mp4 /content/audio.wav /content/labels.json
from google.colab import files; files.download("/content/amman_through_time.mp4")